In [1]:
import sys
import numpy as np
import matplotlib.pyplot as plt

sys.path.append("/mnt/lareaulab/reliscu/code")

from parse_gtf import *

In [2]:
ctype_abund_df = pd.read_csv("data/ctype_abundance/GTEx_cortex_counts_TMMF_All_501_outliers_removed_47840genes_cleaned_44846genes_cleaned_mergeParam0.85_subsetCutoff1.427_Modules_top_corr_enriched_w_Claude_marker_genes_PC1_ctype_abundance.csv", index_col=0)

In [3]:
# Parse GTF attribute column
gtf_file = "/mnt/lareaulab/reliscu/data/GENCODE/GRCh38/gencode.v46.annotation.gtf"
gtf = gtf_parse(gtf_file)
gtf_subset = gtf.loc[gtf['feature'].isin(["gene"])]
attrs = gtf_subset["attribute"].apply(extract_attributes)
attrs_df = attrs.apply(pd.Series)
gtf_parsed = pd.concat([gtf_subset.drop(columns=["attribute"]), attrs_df], axis=1)
gtf_parsed['gene_id'] = gtf_parsed['gene_id'].str.split(".").str[0]

# Gene expr

In [4]:
bulk_expr = pd.read_csv("data/cleaned/TopModPosBC/GTEx_cortex_counts_TMMF_All_501_outliers_removed_47840genes_cleaned_44846genes_cleaned.csv")
bulk_expr.columns.values[0] = "Gene"

In [5]:
mean_expr = pd.DataFrame({
    'Index': range(len(bulk_expr)),
    'Gene': bulk_expr.iloc[:, 0],
    'Expr': bulk_expr.iloc[:, 1:].mean(axis=1)
})

keep = mean_expr.loc[mean_expr.groupby('Gene')['Expr'].idxmax(), 'Index']
bulk_expr_filtered = bulk_expr.iloc[keep.values]

In [6]:
# Filter 0 genes

mask = bulk_expr_filtered.iloc[:, 1:].mean(axis=1) > 0
bulk_expr_filtered = bulk_expr_filtered[mask]

In [8]:
bulk_expr_filtered.shape

(43260, 501)

In [7]:
bulk_expr_filtered = bulk_expr_filtered.set_index("Gene")
common = bulk_expr_filtered.columns.intersection(ctype_abund_df.index)
bulk_expr_filtered = bulk_expr_filtered[common]
ctype_abund_df = ctype_abund_df.loc[common]

In [9]:
# Correlate each cell type with all PSI events across samples

# expr_numeric = bulk_expr_filtered.iloc[:, 1:].apply(pd.to_numeric, errors='coerce')

expr_corr_results = {}
for ct in ctype_abund_df.columns:
    expr_corr_results[ct] = bulk_expr_filtered.T.corrwith(ctype_abund_df[ct])

In [10]:
expr_corr_df = pd.DataFrame(expr_corr_results)
expr_corr_df.head()

,All GABAergic,OPC,All Neuronal,Oligo,Micro/PVM,Endo,VLMC,Astro,All Glutamatergic,Sst,...,L6 IT,Lamp5,Vip,MGE Class,Pax6,L6b,L5/6 NP,Chandelier,L4 IT,L6 IT Car3
Gene,,,,,,,,,,,,,,,,,,,,,
5S_rRNA,0.111217,-0.018700,0.170264,-0.067847,0.119277,-0.016369,-0.060981,0.019281,0.156501,0.111217,...,0.139057,0.082710,0.084952,0.112783,0.112783,0.088972,0.132262,0.156501,0.168495,0.038455
5_8S_rRNA,0.127350,0.016182,0.089924,-0.017090,0.074888,-0.001039,-0.026247,0.028613,0.063030,0.127350,...,0.052076,0.102061,0.127669,0.135387,0.135387,0.078727,0.126005,0.063030,0.083164,0.070268
7SK,0.063752,0.027099,0.049103,0.027167,-0.035397,-0.014747,0.102745,0.039433,0.048891,0.063752,...,0.043591,0.032830,0.064446,0.065549,0.065549,0.043007,0.077193,0.048891,0.040497,0.021780
A1BG,0.528876,0.446496,0.510688,0.202342,0.234522,0.107853,-0.036635,0.400683,0.432919,0.528876,...,0.291727,0.439722,0.597146,0.548136,0.548136,0.512808,0.577304,0.432919,0.505253,0.487382
A1CF,0.132369,-0.002212,0.190976,0.021480,-0.016529,0.009256,-0.044235,0.002307,0.198389,0.132369,...,0.157151,0.086446,0.101142,0.130932,0.130932,0.104208,0.124389,0.198389,0.191877,0.058900


In [11]:
expr_corr_df.to_csv(f"data/corrs/GTEx_cortex_counts_TMMF_All_501_outliers_removed_47840genes_cleaned_44846genes_cleaned_mergeParam0.85_subsetCutoff1.427_Modules_top_corr_enriched_w_Claude_marker_genes_PC1_ctype_abundance_gene_expr_corr.csv")